# Linear SVM and Softmax classifiers

University of Michigan EECS 498/598, Winter 2022. Compact companion to the completed course assignment.

**Static validation only; not executed.** No ordinary-notebook outputs or scores are copied here. Run cells in order only when execution is intended; training variants retain the ordinary notebook parameters. Existing course dependencies and data are required; no installation or repository cloning is performed. Results are isolated under `core_results/linear_classifier`.

## Setup

In [ ]:
from pathlib import Path
import os
import sys
import importlib

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/drive/MyDrive/umich-dlcv-wi2022')]
PROJECT_ROOT = next((p for p in candidates if (p / 'assignments' / 'a2' / 'linear_classifier_core.ipynb').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open the canonical project or assignment directory; mount Drive first in Colab.')
ASSIGNMENT_DIR = PROJECT_ROOT / 'assignments' / 'a2'
if 'eecs598' in sys.modules and Path(sys.modules['eecs598'].__file__).resolve().parent != (ASSIGNMENT_DIR / 'eecs598').resolve():
    raise RuntimeError('Restart the kernel before switching assignments: eecs598 differs across assignments.')
os.chdir(ASSIGNMENT_DIR)
sys.path.insert(0, str(ASSIGNMENT_DIR))
sys.modules.setdefault('imp', importlib)
import torch
import matplotlib.pyplot as plt
import eecs598
from eecs598 import reset_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS_DIR = ASSIGNMENT_DIR / 'core_results' / 'linear_classifier'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed(0)
print('Device:', device)


## Data

In [ ]:
DATA_DIR = next((p for p in (ASSIGNMENT_DIR, PROJECT_ROOT / 'assignments' / 'a1') if (p / 'cifar-10-batches-py').is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Prepare the course CIFAR-10 cache before running this notebook.')
try:
    os.chdir(DATA_DIR)
    data_dict = eecs598.data.preprocess_cifar10(cuda=device == 'cuda', dtype=torch.float32, flatten=True, bias_trick=True, show_examples=False)
finally:
    os.chdir(ASSIGNMENT_DIR)


## Train and select SVM / Softmax on validation data

In [ ]:
from linear_classifier_core import LinearSVM, Softmax, svm_get_search_params, softmax_get_search_params, test_one_param_set
best_models = {}
search_results = {}
for name, cls, get_params in [('svm', LinearSVM, svm_get_search_params), ('softmax', Softmax, softmax_get_search_params)]:
    best_val = -1.0
    search_results[name] = {}
    learning_rates, regularization_strengths = get_params()
    for lr in learning_rates:
        for reg in regularization_strengths:
            reset_seed(0)
            model, train_acc, val_acc = test_one_param_set(cls(), data_dict, lr, reg, num_iters=2000)
            search_results[name][(lr, reg)] = (train_acc, val_acc)
            if val_acc > best_val:
                best_models[name], best_val = model, val_acc
    best_models[name].save(str(RESULTS_DIR / f'{name}_best_model.pt'))
    print(name, 'best validation accuracy (%):', best_val)


## Evaluate and visualize class weights

In [ ]:
for name, model in best_models.items():
    predictions = model.predict(data_dict['X_test'])
    print(name, 'test accuracy (%):', 100 * (predictions == data_dict['y_test']).float().mean().item())
    weights = model.W[:-1].reshape(3, 32, 32, 10).permute(3, 1, 2, 0).cpu()
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for index, ax in enumerate(axes.flat):
        image = weights[index]
        ax.imshow((image - image.min()) / (image.max() - image.min()).clamp_min(1e-12))
        ax.set_title(str(index))
        ax.axis('off')
    fig.suptitle(name)
    fig.savefig(RESULTS_DIR / f'{name}_weights.png', bbox_inches='tight')
    plt.show()
torch.save(search_results, RESULTS_DIR / 'search_results.pt')
